If you want to handle JSON data from A–Z in PySpark, the process generally involves:

1. Reading JSON (from files, streams, or columns)
2. Defining and applying schemas (for performance and correctness)
3. Parsing nested JSON
4. Flattening complex structures
5. Writing JSON back

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType,StructField, StringType,IntegerType,ArrayType
spark = SparkSession.builder.appName('JsonExamples').getOrCreate()


In [0]:
# 2. Example JSON data (could be from file, Kafka, etc.)
json_data = [
    ('{"name":"Alice","age":30,"skills":["Python","Spark"],"address":{"city":"NY","zip":"10001"}}',),
    ('{"name":"Bob","age":25,"skills":["Java","Scala"],"address":{"city":"SF","zip":"94105"}}',)
]

In [0]:
# Create DataFrame with JSON strings
df = spark.createDataFrame(json_data, ["json_str"])
display(df)


In [0]:
# 3. Define schema for JSON
schema = StructType([
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("skills", ArrayType(StringType()), True),
    StructField("address", StructType([
        StructField("city", StringType(), True),
        StructField("zip", StringType(), True)
    ]), True),
    StructField("_corrupt_record",StringType(),True)
])

In [0]:
# 4. Parse JSON string into structured columns
df_parsed = df.withColumn("data", F.from_json(F.col("json_str"), schema))


In [0]:
# 5. Flatten nested fields
df_flat =  df_parsed.select(F.col("data.name"),F.col("data.age"),F.explode(F.col("data.skills")).alias("skill"),F.col("data.address.city"),F.col("data.address.zip").alias("zip"))


In [0]:
#show results
display(df_flat)

In [0]:
df_flat.write.mode("overwrite").json("/Workspace/Users/pdkusalkar@gmail.com/databricks_work/handling_json_data/json_output")

##### What This Does
- Reads JSON from a column (json_str).
- Defines a schema to avoid expensive schema inference.
- Parses JSON into structured columns using from_json.
- Flattens nested objects (address.city) and arrays (skills with explode).
- Writes JSON back to disk.

#### Key Functions for JSON in PySpark
- Function	Purpose
- spark.read.json(path)	Read JSON files directly into a DataFrame
- from_json(col, schema)	Parse JSON string column into structured data
- to_json(col)	Convert struct/array column back to JSON string
- explode(array_col)	Flatten arrays into multiple rows
- get_json_object(col, path)	Extract a specific JSON field using JSONPath
- schema_of_json(json_string)	Infer schema from a JSON sample

✅ Tip: Always define a schema for large datasets — it’s much faster than letting Spark infer it.